In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("C:/Users/marce/Documents/Síntese Jr/Dados/Projeto Trainee/data.csv")
LABELS_PATH = Path("C:/Users/marce/Documents/Síntese Jr/Dados/Projeto Trainee/labels.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError("Defina o caminho para data.csv")

if not LABELS_PATH.exists():
    raise FileNotFoundError("Defina o caminho para labels.csv")

df_data = pd.read_csv(DATA_PATH)
df_labels = pd.read_csv(LABELS_PATH)

In [2]:
#concatenando os dois data frames
df_completo = pd.concat([df_data, df_labels], axis=1)

In [3]:
#vendo se há linhas duplicadas
total_duplicatas = df_completo.duplicated().sum()
print(f'Total de linhas duplicadas: {total_duplicatas}')

Total de linhas duplicadas: 0


In [4]:
#descobrindo quantos dados nulos temos:
total_nulos = df_completo.isnull().sum().sum()
print(f'Total de dados nulos: {total_nulos}')

Total de dados nulos: 0


In [5]:
#estabalecendo uma variancia minima dos genes para reduzir possiveis ruidos no modelo
variancias = df_completo.var(numeric_only=True)
colunas_para_manter = variancias[variancias > 0.1].index.tolist()
colunas_para_manter.append('Class')
df_final = df_completo[colunas_para_manter]
print(f"Total de colunas ANTES do corte: {df_completo.shape[1]}")
print(f"Total de colunas DEPOIS do corte: {df_final.shape[1]}")

Total de colunas ANTES do corte: 20534
Total de colunas DEPOIS do corte: 19279


In [6]:
df_teste = df_final.select_dtypes(include=["number"])
#Análise dos quartis e dos outliers
outliers = 0
 
mediana = df_teste.median()
Q1 = df_teste.quantile(0.25)
Q3 = df_teste.quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - (1.5 * IQR)
limite_superior = Q3 + (1.5 * IQR)
outliers = (df_teste < limite_inferior) | (df_teste > limite_superior)
qtd_outliers = outliers.sum().sum()

display(f"Total de outliers: {qtd_outliers}")
    


'Total de outliers: 426270'

In [7]:
#Utilizando os quartis para retirar os outliers e substituir os outliers pela mediana
colunas  = df_final.select_dtypes(include=["number"]).columns

medianas = df_final[colunas].median()
nao_outlier = (df_final[colunas] >= limite_inferior) & (df_final[colunas] <= limite_superior)
df_final[colunas] = df_final[colunas].where(nao_outlier, medianas, axis=1)

outliers = (df_final[colunas] < limite_inferior) | (df_final[colunas] > limite_superior)
qtd_outliers = outliers.sum().sum()
print(f"Total outliers: {qtd_outliers}")



C:\Users\marce\AppData\Local\Temp\ipykernel_16016\1171099819.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final[colunas] = df_final[colunas].where(nao_outlier, medianas, axis=1)


Total outliers: 0


Para as inconsistências que podem levar a enviezamento, foram verificados os nulos e duplicados, não há nenhuma contradição pois as colunas contem apenas números, classes e o id do gene e não há vazamento de dados pois nenhuma variável é gerada como resultado.

analise exploratória de dados:
    será feita através de 3 passos:
        1: criação de dataframes exclusivos para cada tipo de tumor;
        2: análise dos 10 genes mais expressivos e menos expressivos para cada tipo de tumor;
        3: proporção de cada tipo de tumor da amostra.

In [8]:
#Passo 1: Criando dataframes para cada tipo de tumor
df_final = df_final.set_index("Class")
df_prad = df_final.loc["PRAD"]
df_luad = df_final.loc["LUAD"]
df_brca = df_final.loc["BRCA"]
df_coad = df_final.loc["COAD"]
df_kirc = df_final.loc["KIRC"]

In [9]:
#para os tumores PRAD:
prad_10maiores = df_prad.apply(lambda linha: linha.nlargest(10).index.tolist(), axis=1)
prad_10menores = df_prad.apply(lambda linha: linha.nsmallest(10).index.tolist(), axis=1)
todos_os_genes_maiores_prad = prad_10maiores.explode()
todos_os_genes_menores_prad = prad_10menores.explode()
frequencia_maiores_prad = todos_os_genes_maiores_prad.value_counts()
frequencia_menores_prad = todos_os_genes_menores_prad.value_counts()
frequencia_maiores_prad = frequencia_maiores_prad.head(10)
frequencia_menores_prad = frequencia_menores_prad.head(10)

#para os tumores LUAD:  
luad_10maiores = df_luad.apply(lambda linha: linha.nlargest(10).index.tolist(), axis=1)
luad_10menores = df_luad.apply(lambda linha: linha.nsmallest(10).index.tolist(), axis=1)
todos_os_genes_maiores_luad = luad_10maiores.explode()
todos_os_genes_menores_luad = luad_10menores.explode()
frequencia_maiores_luad = todos_os_genes_maiores_luad.value_counts()
frequencia_menores_luad = todos_os_genes_menores_luad.value_counts()
frequencia_maiores_luad = frequencia_maiores_luad.head(10)
frequencia_menores_luad = frequencia_menores_luad.head(10)

#para os tumores BRCA:
brca_10maiores = df_brca.apply(lambda linha: linha.nlargest(10).index.tolist(), axis=1)
brca_10menores = df_brca.apply(lambda linha: linha.nsmallest(10).index.tolist(), axis=1)
todos_os_genes_maiores_brca = brca_10maiores.explode() 
todos_os_genes_menores_brca = brca_10menores.explode()
frequencia_maiores_brca = todos_os_genes_maiores_brca.value_counts()
frequencia_menores_brca = todos_os_genes_menores_brca.value_counts()
frequencia_maiores_brca = frequencia_maiores_brca.head(10)
frequencia_menores_brca = frequencia_menores_brca.head(10)

#para os tumores COAD:
coad_10maiores = df_coad.apply(lambda linha: linha.nlargest(10).index.tolist(), axis=1)
coad_10menores = df_coad.apply(lambda linha: linha.nsmallest(10).index.tolist(), axis=1)
todos_os_genes_maiores_coad = coad_10maiores.explode()
todos_os_genes_menores_coad = coad_10menores.explode()
frequencia_maiores_coad = todos_os_genes_maiores_coad.value_counts()
frequencia_menores_coad = todos_os_genes_menores_coad.value_counts()
frequencia_maiores_coad = frequencia_maiores_coad.head(10)
frequencia_menores_coad = frequencia_menores_coad.head(10)

#para os tumores KIRC:
kirc_10maiores = df_kirc.apply(lambda linha: linha.nlargest(10).index.tolist(), axis=1)
kirc_10menores = df_kirc.apply(lambda linha: linha.nsmallest(10).index.tolist(), axis=1)
todos_os_genes_maiores_kirc = kirc_10maiores.explode()
todos_os_genes_menores_kirc = kirc_10menores.explode()
frequencia_maiores_kirc = todos_os_genes_maiores_kirc.value_counts()
frequencia_menores_kirc = todos_os_genes_menores_kirc.value_counts()
frequencia_maiores_kirc = frequencia_maiores_kirc.head(10)
frequencia_menores_kirc = frequencia_menores_kirc.head(10)

Passo 2: Criando um dataframe para visualizar melhor quais genes são mais expressivos e menos expressivos para cada tipo de tumor:

In [10]:
df_frequencia_maiores_luad = frequencia_maiores_luad.reset_index()
df_frequencia_maiores_luad.columns = ['Gene', 'Frequencia']

df_frequencia_menores_luad = frequencia_menores_luad.reset_index()
df_frequencia_menores_luad.columns = ['Gene', 'Frequencia']

df_frequencia_maiores_prad = frequencia_maiores_prad.reset_index()
df_frequencia_maiores_prad.columns = ['Gene', 'Frequencia']

df_frequencia_menores_prad = frequencia_menores_prad.reset_index()
df_frequencia_menores_prad.columns = ['Gene', 'Frequencia']

df_frequencia_maiores_brca = frequencia_maiores_brca.reset_index()
df_frequencia_maiores_brca.columns = ['Gene', 'Frequencia']

df_frequencia_menores_brca = frequencia_menores_brca.reset_index()
df_frequencia_menores_brca.columns = ['Gene', 'Frequencia']

df_frequencia_maiores_coad = frequencia_maiores_coad.reset_index()
df_frequencia_maiores_coad.columns = ['Gene', 'Frequencia']

df_frequencia_menores_coad = frequencia_menores_coad.reset_index()
df_frequencia_menores_coad.columns = ['Gene', 'Frequencia']

df_frequencia_maiores_kirc = frequencia_maiores_kirc.reset_index()
df_frequencia_maiores_kirc.columns = ['Gene', 'Frequencia']

df_frequencia_menores_kirc = frequencia_menores_kirc.reset_index()
df_frequencia_menores_kirc.columns = ['Gene', 'Frequencia']

In [11]:
comparativo_maiores = pd.concat(
    {
        "LUAD/Pulmao": frequencia_maiores_luad,
        "PRAD/Prostata": frequencia_maiores_prad,
        "BRCA/Mama": frequencia_maiores_brca,
        "COAD/Colon": frequencia_maiores_coad,
        "KIRC/Rim": frequencia_maiores_kirc,
    },
    axis=1,
) 

In [12]:
comparativo_menores = pd.concat(
    {
        "LUAD/Pulmao": frequencia_menores_luad,
        "PRAD/Prostata": frequencia_menores_prad,
        "BRCA/Mama": frequencia_menores_brca,
        "COAD/Colon": frequencia_menores_coad,
        "KIRC/Rim": frequencia_menores_kirc,
    },
    axis=1,
) 

Passo 3: Proporção dos tipos de tumores e seus respectivos genes expressivos 

In [13]:
print(df_luad.shape)
print(df_prad.shape)
print(df_brca.shape)
print(df_coad.shape)
print(df_kirc.shape)

display (comparativo_maiores)

(141, 19278)
(136, 19278)
(300, 19278)
(78, 19278)
(146, 19278)


,LUAD/Pulmao,PRAD/Prostata,BRCA/Mama,COAD/Colon,KIRC/Rim
gene_230,137.0,112.0,265.0,75.0,111.0
gene_289,92.0,NaN,NaN,NaN,NaN
gene_6698,86.0,NaN,NaN,NaN,53.0
gene_232,81.0,93.0,216.0,72.0,NaN
gene_5380,77.0,135.0,224.0,71.0,129.0
gene_3371,72.0,NaN,NaN,NaN,97.0
gene_6857,52.0,NaN,88.0,58.0,127.0
gene_4041,50.0,NaN,216.0,28.0,NaN
gene_10194,49.0,NaN,NaN,NaN,NaN
gene_6566,48.0,NaN,190.0,NaN,NaN


Especifico:

.Pulmao: Gene 289 é expressivo apenas para o pulmao - 65% de presenca
.Prostata: gene_15242 é expressivo apenas para a prostata - 78% de presenca
.Mama: gene_4041 é muito expressivo para a mama - 72%. gene_4042 é muito expressivo APENAS para a mama 61%- 
.Cólon: gene_3540 é muito expressivo apenas para o cólon - 63%
.Rim: aproximadamente 88% das amostras possuem o gene 5380 e 6857 sendo os mais expressivos

Geral:

.O gene 230 é o unico expressivamente ativo nos 5 tumores, sendo um forte indicador de precursor de celulas cancerigenas 

In [14]:
#genes menos expressivos em cada tipo de tumor

print(df_luad.shape)
print(df_prad.shape)
print(df_brca.shape)
print(df_coad.shape)
print(df_kirc.shape)

display (comparativo_menores)

(141, 19278)
(136, 19278)
(300, 19278)
(78, 19278)
(146, 19278)


,LUAD/Pulmao,PRAD/Prostata,BRCA/Mama,COAD/Colon,KIRC/Rim
gene_24,141.0,136.0,300.0,78.0,146.0
gene_37,126.0,129.0,268.0,52.0,144.0
gene_14,117.0,107.0,216.0,68.0,85.0
gene_13,113.0,113.0,236.0,73.0,104.0
gene_30,103.0,80.0,247.0,NaN,NaN
gene_41,95.0,110.0,200.0,NaN,138.0
gene_25,87.0,NaN,186.0,53.0,NaN
gene_43,76.0,92.0,159.0,NaN,119.0
gene_31,73.0,NaN,157.0,NaN,74.0
gene_10,70.0,NaN,178.0,62.0,NaN


o gene 24 nao é expressivo em absolutamente nenhuma amostra

os genes 37, 14 e 13 lideram, após o gene 24, como os menos expressivos, nessa respectiva ordem

In [16]:
print(df_luad.shape)
print(df_prad.shape)
print(df_brca.shape)
print(df_coad.shape)
print(df_kirc.shape)

(141, 19278)
(136, 19278)
(300, 19278)
(78, 19278)
(146, 19278)


17,60% das amostras sao de LUAD - Pulmao                       
17% das amostras sao de prad - prostata                         
37,4% das amostras sao de brca -mama                             
9,7% das amostras sao de coad - colon                               
18,22% das amostras sao de kirc - rim                            